In [1]:
import os
import statistics
from collections import Counter
from datetime import datetime
from json import dumps

import pymongo
from dotenv import dotenv_values


In [2]:
# 1. Chargement ciblé des variables d'environnement
env_vars = dotenv_values(".env")
env_local_vars = dotenv_values(".env.local")

atlas_uri = env_vars.get("ATLAS_URI")
local_uri = env_local_vars.get("LOCAL_URI")

# 2. Test de la connexion locale
try:
    if not local_uri:
        raise ValueError("LOCAL_URI non trouvé dans .env.local")
    local_client = pymongo.MongoClient(local_uri, serverSelectionTimeoutMS=5000)
    local_client.admin.command("ping")
    print("Connexion locale établie avec succès")
except Exception as e:
    print(f"Erreur de connexion locale : {e}")

# 3. Test de la connexion cloud (Atlas)
try:
    if not atlas_uri:
        raise ValueError("ATLAS_URI non trouvé dans .env")
    atlas_client = pymongo.MongoClient(atlas_uri, serverSelectionTimeoutMS=5000)
    atlas_client.admin.command("ping")
    print("Connexion cloud établie avec succès")
except Exception as e:
    print(f"Erreur de connexion cloud : {e}")

Connexion locale établie avec succès


Connexion cloud établie avec succès


## CRUD Python (livrable 3)

Les quatre opérations sur `securite_routiere.accidents`, avec une gestion d'erreurs réelle (`pymongo.errors`). La connexion (`atlas_client`) réutilise celle établie plus haut à partir de `.env` — aucun identifiant codé en dur ici.

In [3]:
from pymongo.errors import DuplicateKeyError, PyMongoError

accidents = atlas_client["securite_routiere"]["accidents"]

In [4]:
def create_accident(document: dict):
    """Insère un nouvel accident. Retourne l'_id inséré, ou None en cas d'échec."""
    try:
        result = accidents.insert_one(document)
        print(f"Inséré avec _id={result.inserted_id}")
        return result.inserted_id
    except DuplicateKeyError:
        print("Échec : un document avec cet _id existe déjà")
    except PyMongoError as exc:
        print(f"Échec de l'insertion : {exc}")
    return None


nouvel_accident = {
    "Num_Acc": 202400000001,
    "jour": 1, "mois": 1, "an": 2024, "hrmn": "00:00",
    "localisation": {"type": "Point", "coordinates": [2.3522, 48.8566]},
    "lieu": {"catr": 3, "voie": "TEST", "vma": 50},
    "vehicules": [],
    "dep": "75", "com": "75056", "agg": 1, "int": 1, "atm": 1, "lum": 1, "col": 1,
}
create_accident(nouvel_accident)

Inséré avec _id=6a914faf1eacda51b368567f


ObjectId('6a914faf1eacda51b368567f')

In [5]:
def read_accidents(filtre: dict, limite: int = 5) -> list:
    """Recherche des accidents selon un filtre. Retourne une liste vide en cas d'échec."""
    try:
        return list(accidents.find(filtre).limit(limite))
    except PyMongoError as exc:
        print(f"Échec de la lecture : {exc}")
        return []


resultats = read_accidents({"dep": "75"})
for doc in resultats:
    print(doc["Num_Acc"], doc.get("dep"), doc.get("lieu", {}).get("voie"))

202400000011 75 PORTE DE VINCENNES
202400000012 75 PORTE DE CLICHY
202400000013 75 RUE GASTON TESSIER
202400000014 75 RUE DE THIONVILLE
202400000015 75 QUAI DE LA GIRONDE


In [6]:
def update_accident(num_acc: int, changements: dict) -> int:
    """Met à jour un accident par son Num_Acc. Retourne le nombre de documents modifiés."""
    try:
        result = accidents.update_one({"Num_Acc": num_acc}, {"$set": changements})
        if result.matched_count == 0:
            print(f"Aucun accident trouvé avec Num_Acc={num_acc}")
        else:
            print(f"{result.modified_count} document(s) modifié(s)")
        return result.modified_count
    except PyMongoError as exc:
        print(f"Échec de la mise à jour : {exc}")
        return 0


update_accident(202400000001, {"adr": "Adresse corrigée"})

0 document(s) modifié(s)


0

In [7]:
def delete_accident(num_acc: int, filtre_supplementaire: dict = None) -> int:
    """Supprime un accident par son Num_Acc. Retourne le nombre de documents supprimés."""
    filtre = {"Num_Acc": num_acc}
    if filtre_supplementaire:
        filtre.update(filtre_supplementaire)
    try:
        result = accidents.delete_one(filtre)
        if result.deleted_count == 0:
            print(f"Aucun accident trouvé avec Num_Acc={num_acc}")
        else:
            print(f"{result.deleted_count} document(s) supprimé(s)")
        return result.deleted_count
    except PyMongoError as exc:
        print(f"Échec de la suppression : {exc}")
        return 0


delete_accident(999999999999)

Aucun accident trouvé avec Num_Acc=999999999999


0

### Démonstration : gestion des erreurs

Pour prouver que la gestion d'erreurs n'est pas cosmétique, on déclenche volontairement une vraie erreur pymongo pour chacune des 4 opérations. Dans les quatre cas, le programme ne plante pas : la fonction affiche le message d'erreur et retourne une valeur neutre (`None`, `[]` ou `0`).

In [8]:
doublon = {"_id": "test-duplicate-id", "Num_Acc": 888888888888}
create_accident(doublon)
create_accident(doublon)

accidents.delete_one({"_id": "test-duplicate-id"})

Inséré avec _id=test-duplicate-id
Échec : un document avec cet _id existe déjà


DeleteResult({'n': 1, 'electionId': ObjectId('7fffffff00000000000000ed'), 'opTime': {'ts': Timestamp(1787908015, 8), 't': 237}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1787908015, 8), 'signature': {'hash': b'\x89\xd2Vne[\x90\xe3\x7f\xcb\xc3\xef\x0bF\xc7\xdf\x8b\xb1I.', 'keyId': 7643078544444096515}}, 'operationTime': Timestamp(1787908015, 8)}, acknowledged=True)

In [9]:
resultat = read_accidents({"dep": {"$operateurInexistant": "75"}})
print("Valeur renvoyée malgré l'erreur :", resultat)

Échec de la lecture : unknown operator: $operateurInexistant, full error: {'ok': 0.0, 'errmsg': 'unknown operator: $operateurInexistant', 'code': 2, 'codeName': 'BadValue', '$clusterTime': {'clusterTime': Timestamp(1787908015, 8), 'signature': {'hash': b'\x89\xd2Vne[\x90\xe3\x7f\xcb\xc3\xef\x0bF\xc7\xdf\x8b\xb1I.', 'keyId': 7643078544444096515}}, 'operationTime': Timestamp(1787908015, 8)}
Valeur renvoyée malgré l'erreur : []


In [10]:
update_accident(202400000011, {"_id": "nouvelle-valeur"})

Échec de la mise à jour : Performing an update on the path '_id' would modify the immutable field '_id', full error: {'index': 0, 'code': 66, 'errmsg': "Performing an update on the path '_id' would modify the immutable field '_id'"}


0

In [11]:
delete_accident(202400000011, filtre_supplementaire={"dep": {"$operateurInexistant": "75"}})

Échec de la suppression : unknown operator: $operateurInexistant, full error: {'index': 0, 'code': 2, 'errmsg': 'unknown operator: $operateurInexistant'}


0

## Administration — sauvegarde et restauration (livrable 6)

Scripts de sauvegarde (`mongodump`) et de restauration (`mongorestore`) de la base `securite_routiere` (collection `accidents`, modèle dénormalisé retenu pour le projet) sur Atlas, avec une sauvegarde automatisée programmée chaque vendredi (cron).

In [12]:
import subprocess
from datetime import datetime
from pathlib import Path

env = {**dotenv_values(".env.local"), **dotenv_values(".env")}
ATLAS_URI = env.get("ATLAS_URI")
LOCAL_URI = env.get("LOCAL_URI")
DB_NAME = "securite_routiere"
BACKUP_DIR = Path("backups")
BACKUP_DIR.mkdir(exist_ok=True)


def backup_database(uri: str = ATLAS_URI, db_name: str = DB_NAME, backup_dir: Path = BACKUP_DIR) -> Path:
    """Sauvegarde la base via mongodump dans un dossier horodaté."""
    if not uri:
        raise ValueError("URI manquante dans .env / .env.local")

    timestamp = datetime.now().strftime("%Y-%m-%d_%Hh%M")
    target = backup_dir / f"{db_name}_{timestamp}"

    result = subprocess.run(
        ["mongodump", f"--uri={uri}", f"--db={db_name}", f"--out={target}"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"mongodump a échoué :\n{result.stderr}")

    print(f"Sauvegarde créée : {target}")
    return target

In [13]:
def restore_database(dump_path: Path, uri: str = ATLAS_URI, db_name: str = DB_NAME, drop: bool = False) -> None:
    """Restaure une sauvegarde mongodump vers la base cible."""
    if not uri:
        raise ValueError("URI manquante dans .env / .env.local")

    dump_path = Path(dump_path)
    source = dump_path / db_name
    if not source.exists():
        raise FileNotFoundError(f"Aucun dump trouvé dans {source}")

    cmd = ["mongorestore", f"--uri={uri}", f"--nsInclude={db_name}.*", str(dump_path)]
    if drop:
        cmd.insert(1, "--drop")

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"mongorestore a échoué :\n{result.stderr}")

    print(f"Restauration terminée depuis : {dump_path}")

### Démonstration : sauvegarde

On sauvegarde `securite_routiere` depuis Atlas avec `mongodump`.

In [14]:
dump_path = backup_database(uri=ATLAS_URI)

Sauvegarde créée : backups/securite_routiere_2026-08-28_11h06


### Démonstration : restauration

On restaure ce dump vers MongoDB local (`LOCAL_URI`) pour vérifier son intégrité, sans jamais toucher aux données sur Atlas.

In [15]:
restore_database(dump_path, uri=LOCAL_URI, drop=True)

Restauration terminée depuis : backups/securite_routiere_2026-08-28_11h06


### Automatisation : sauvegarde chaque vendredi

Un notebook ne s'exécute pas tout seul un jour donné : l'automatisation passe par un **script autonome** (`scripts/weekly_backup.py`) déclenché par une tâche planifiée (`cron` sous macOS/Linux, ou le Planificateur de tâches sous Windows).

On génère ce script ci-dessous, puis on l'enregistre dans une tâche cron qui tourne chaque vendredi à 20h.

In [16]:
Path("scripts").mkdir(exist_ok=True)

In [17]:
%%writefile scripts/weekly_backup.py
#!/usr/bin/env python3
"""Sauvegarde hebdomadaire de la base securite_routiere. A lancer via cron chaque vendredi."""
import subprocess
import sys
from datetime import datetime
from pathlib import Path

from dotenv import dotenv_values

ROOT = Path(__file__).resolve().parent.parent
env = {**dotenv_values(ROOT / ".env.local"), **dotenv_values(ROOT / ".env")}
ATLAS_URI = env.get("ATLAS_URI")
DB_NAME = "securite_routiere"
BACKUP_DIR = ROOT / "backups"
BACKUP_DIR.mkdir(exist_ok=True)


def main() -> int:
    if not ATLAS_URI:
        print("ATLAS_URI manquant dans .env / .env.local", file=sys.stderr)
        return 1

    timestamp = datetime.now().strftime("%Y-%m-%d_%Hh%M")
    target = BACKUP_DIR / f"{DB_NAME}_{timestamp}"

    result = subprocess.run(
        ["mongodump", f"--uri={ATLAS_URI}", f"--db={DB_NAME}", f"--out={target}"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"mongodump a échoué :\n{result.stderr}", file=sys.stderr)
        return 1

    print(f"Sauvegarde créée : {target}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Overwriting scripts/weekly_backup.py


### Planifier la tâche cron (à faire une fois, dans le terminal)

Ouvrir l'éditeur crontab :
```bash
crontab -e
```

Ajouter cette ligne (chaque vendredi à 20h00) :
```cron
0 20 * * 5 cd /Users/latr/Desktop/projetNosql/ipssi-nosql && /opt/homebrew/bin/uv run python scripts/weekly_backup.py >> backups/backup.log 2>&1
```

- `0 20 * * 5` : minute=0, heure=20, tous les jours du mois, tous les mois, jour de semaine=5 (vendredi, 0=dimanche)
- Les sorties (succès ou erreur) sont ajoutées à `backups/backup.log`, à consulter pour vérifier que la sauvegarde a bien tourné
- Vérifier la tâche enregistrée avec `crontab -l`